# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>

In [21]:
from litellm import completion
from dotenv import load_dotenv
import json
#from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

True

# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [22]:
LITE_MODE = True

In [23]:
username = "imphe1218"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 22,000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Wei

In [25]:
items[2].id

In [26]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [27]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [28]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [29]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages,  model="openai/local-model",api_base="http://127.0.0.1:8081/v1",
    api_key="not-needed")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
#print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents") commenting out because im running local AI model


Title: Schlage F59 & 613 Andover Interior Half-Knob with Deadbolt

Category: Electronics

Brand: Schlage

Description: A high-security interior half-knob with a deadbolt for front doors, finished in oil-rubbed bronze.

Details: Precision engineered for 100% solid construction, easy to install, lifetime mechanical and finish warranty.

Input tokens: 377
Output tokens: 76


In [ ]:

#messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
#response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
#print(response.choices[0].message.content)
#print()
#print(f"Input tokens: {response.usage.prompt_tokens}")
#print(f"Output tokens: {response.usage.completion_tokens}")
#print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")
# not running since we did this already from the preceeding notebook

In [30]:
#MODEL = "openai/gpt-oss-20b"
MODEL="openai/local-model"


In [31]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [33]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [34]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/local-model", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two piece m

In [35]:

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [36]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [38]:
#import os
#from groq import Groq

#groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [39]:

#with open("jsonl/0_1000.jsonl", "rb") as f:
#    response = groq.files.create(file=f, purpose="batch")
#response

In [40]:
#file_id = response.id
#file_id

In [41]:
#response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
#response

In [42]:
#result = groq.batches.retrieve(response.id)
#result

In [ ]:
#response = groq.files.content(result.output_file_id)
#response.write_to_file("jsonl/batch_results.jsonl")

In [48]:
import json
from pathlib import Path
from litellm import completion
from tqdm.notebook import tqdm


# ---------------------------------------------------------
# File locations
# ---------------------------------------------------------

INPUT_FILE = Path("jsonl/0_1000.jsonl")
OUTPUT_FOLDER = Path("jsonl")
OUTPUT_FILE = OUTPUT_FOLDER / "batch_results.jsonl"

# Create week6/jasonkhaw if it does not exist.
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# Local llama.cpp server
# ---------------------------------------------------------

MODEL = "openai/microsoft_Phi-4-mini-instruct-Q4_K_M"
API_BASE = "http://127.0.0.1:8081/v1"


# ---------------------------------------------------------
# Read input JSONL and process one line at a time
# ---------------------------------------------------------

with (
    INPUT_FILE.open("r", encoding="utf-8") as input_file,
    OUTPUT_FILE.open("w", encoding="utf-8") as output_file
):
    for line in tqdm(input_file, desc="Processing requests"):

        # Ignore empty lines.
        if not line.strip():
            continue

        request = json.loads(line)

        custom_id = request["custom_id"]
        messages = request["body"]["messages"]

        # Retrieve the system and user content.
        system_content = ""
        user_content = ""

        for message in messages:
            if message["role"] == "system":
                system_content = message["content"]

            elif message["role"] == "user":
                user_content = message["content"]

        # Append system content and user content into one prompt.
        combined_prompt = (
            system_content
            + "\n\n"
            + user_content
        )

        try:
            # Send one request to the local llama.cpp model.
            local_response = completion(
                model=MODEL,
                api_base=API_BASE,
                api_key="local",
                messages=[
                    {
                        "role": "user",
                        "content": combined_prompt
                    }
                ],
                temperature=0,
                max_tokens=500
            )

            # Retrieve the generated text.
            generated_content = (
                local_response
                .choices[0]
                .message
                .content
            )

            # Create a Groq-style successful output object.
            result = {
                "id": f"local-{custom_id}",
                "custom_id": custom_id,
                "response": {
                    "status_code": 200,
                    "request_id": f"local-request-{custom_id}",
                    "body": {
                        "id": f"local-completion-{custom_id}",
                        "model": "microsoft_Phi-4-mini-instruct-Q4_K_M",
                        "choices": [
                            {
                                "index": 0,
                                "message": {
                                    "role": "assistant",
                                    "content": generated_content
                                },
                                "finish_reason": "stop"
                            }
                        ]
                    }
                },
                "error": None
            }

        except Exception as error:

            # Create a Groq-style failed output object.
            result = {
                "id": f"local-{custom_id}",
                "custom_id": custom_id,
                "response": None,
                "error": {
                    "message": str(error)
                }
            }

        # Write one JSON object on one physical line.
        output_file.write(
            json.dumps(result, ensure_ascii=False) + "\n"
        )

        # Immediately save the current result to disk.
        output_file.flush()


print("\nFinished.")
print(f"Results saved to: {OUTPUT_FILE.resolve()}")

Processing requests: 0it [00:00, ?it/s]


Finished.
Results saved to: /home/imphe1218/Projects/llm_engineering/week6/jsonl/batch_results.jsonl


In [ ]:
with open("jsonl/batch_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


NameError: name 'items' is not defined

In [50]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [51]:
print(items[1000].summary)

None


## I've put exactly this logic into a Batch class

- Divides items into groups of 1,000
- Kicks off batches for each
- Allows us to monitor and collect the results when complete

## COSTS

Using Groq, for me - this cost under $1 for the Lite dataset and under $30 for the big dataset

But you don't need to pay anything! In the next lab, you can load my pre-processed results

In [ ]:
Batch.create(items, LITE_MODE)

In [ ]:
Batch.run()

In [ ]:
Batch.fetch()

In [ ]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
print(items[10234].summary)

In [ ]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

In [54]:
from litellm import completion

LOCAL_MODEL = "openai/local-phi-4-mini"
LOCAL_API_BASE = "http://127.0.0.1:8081/v1"


def summarize_locally(item):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": item.full,
        },
    ]

    response = completion(
        model=LOCAL_MODEL,
        api_base=LOCAL_API_BASE,
        api_key="local",
        messages=messages,
        temperature=0.0,
        max_tokens=180,
        timeout=300,
    )

    content = response.choices[0].message.content

    if not content:
        raise RuntimeError("Local model returned an empty response.")

    return content.strip()

In [55]:
print(summarize_locally(items[20_000]))

Title: SAFUEL 10000mAh Magnetic Wireless Power Bank
Category: Electronics
Brand: SAFUEL
Description: Fast-charging 10000mAh power bank with magnetic wireless charging for iPhone 14/13/12 series.
Details: Compatible with MagSafe cases, USB-C in/out, 20W PD fast charging, 3 years warranty, pocket-sized, LED indicator lights.


In [ ]:
from tqdm.notebook import tqdm

# Process any items that still do not have a summary
for i in tqdm(range(len(items))):
    if not getattr(items[i], "summary", None):
        items[i].summary = summarize_locally(items[i])

  0%|          | 0/22000 [00:00<?, ?it/s]

## Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets (in case you decide to use lite later)

In [52]:
username = "imphe1218"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

ValueError: All datasets in `DatasetDict` should have the same features but features for 'train' and 'validation' don't match: {'title': Value(dtype='string', id=None), 'category': Value(dtype='string', id=None), 'price': Value(dtype='float64', id=None), 'full': Value(dtype='string', id=None), 'weight': Value(dtype='float64', id=None), 'summary': Value(dtype='string', id=None), 'prompt': Value(dtype='null', id=None), 'id': Value(dtype='int64', id=None)} != {'title': Value(dtype='string', id=None), 'category': Value(dtype='string', id=None), 'price': Value(dtype='float64', id=None), 'full': Value(dtype='string', id=None), 'weight': Value(dtype='float64', id=None), 'summary': Value(dtype='null', id=None), 'prompt': Value(dtype='null', id=None), 'id': Value(dtype='int64', id=None)}

## And here they are!

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full
